In [ ]:
import pandas as pd

df = pd.read_excel("Online Retail.xlsx")

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

In [ ]:
df.head()

In [ ]:
print("DATA TYPES:")
print(df.dtypes)

print("\nMISSING VALUES:")
print(df.isnull().sum())

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
df[df.duplicated(keep=False)].head(10)

In [ ]:
before = len(df)

df = df.drop_duplicates()

after = len(df)

print("Rows before removing duplicates:", before)
print("Rows after removing duplicates:", after)
print("Duplicates removed:", before - after)
print("Duplicates remaining:", df.duplicated().sum())

In [ ]:
missing_desc = df[df["Description"].isnull()]

print("Rows with missing Description:", len(missing_desc))
print("\nSample of missing Description records:")
display(missing_desc.head(10))

In [ ]:
print("Missing Description records:", len(missing_desc))
print("With UnitPrice = 0:", (missing_desc["UnitPrice"] == 0).sum())
print("With missing CustomerID:", missing_desc["CustomerID"].isnull().sum())
print("With Quantity <= 0:", (missing_desc["Quantity"] <= 0).sum())

In [ ]:
before = len(df)

df = df.dropna(subset=["Description"])

after = len(df)

print("Rows before removing missing descriptions:", before)
print("Rows after:", after)
print("Records removed:", before - after)
print("Missing Description remaining:", df["Description"].isnull().sum())

In [ ]:
print("Total records:", len(df))
print("Missing CustomerID:", df["CustomerID"].isnull().sum())
print("CustomerID available:", df["CustomerID"].notnull().sum())
print("Missing CustomerID %:", round(df["CustomerID"].isnull().mean() * 100, 2))

In [ ]:
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

print("Revenue column created successfully!")
print(df[["Quantity", "UnitPrice", "Revenue"]].head())

In [ ]:
print("QUANTITY CHECK")
print("Quantity <= 0:", (df["Quantity"] <= 0).sum())
print("Minimum Quantity:", df["Quantity"].min())
print("Maximum Quantity:", df["Quantity"].max())

print("\nUNIT PRICE CHECK")
print("UnitPrice <= 0:", (df["UnitPrice"] <= 0).sum())
print("Minimum UnitPrice:", df["UnitPrice"].min())
print("Maximum UnitPrice:", df["UnitPrice"].max())

In [ ]:
negative_qty = df[df["Quantity"] < 0]

print("Negative quantity records:", len(negative_qty))
print("\nSample negative-quantity transactions:")
display(negative_qty[[
    "InvoiceNo", "StockCode", "Description",
    "Quantity", "UnitPrice", "Revenue"
]].head(10))

print("\nInvoice numbers starting with C:",
      negative_qty["InvoiceNo"].astype(str).str.startswith("C").sum())

In [ ]:
negative_price = df[df["UnitPrice"] < 0]

print("Negative UnitPrice records:", len(negative_price))
print("\nSample:")
display(negative_price[[
    "InvoiceNo", "StockCode", "Description",
    "Quantity", "UnitPrice", "Revenue"
]].head(10))

In [ ]:
sales_df = df[
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0)
].copy()

print("Master dataset:", len(df))
print("Sales-analysis dataset:", len(sales_df))
print("Records excluded from sales analysis:", len(df) - len(sales_df))
print("Negative quantities remaining:", (sales_df["Quantity"] <= 0).sum())
print("Non-positive prices remaining:", (sales_df["UnitPrice"] <= 0).sum())

In [ ]:
sales_df[["Quantity", "UnitPrice", "Revenue"]].describe()

In [ ]:
print("SKEWNESS")
print("Quantity:", round(sales_df["Quantity"].skew(), 2))
print("UnitPrice:", round(sales_df["UnitPrice"].skew(), 2))
print("Revenue:", round(sales_df["Revenue"].skew(), 2))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.hist(sales_df["Revenue"], bins=100)

plt.title("Distribution of Revenue")
plt.xlabel("Revenue")
plt.ylabel("Frequency")

plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.hist(sales_df["Revenue"], bins=100)

plt.title("Distribution of Revenue")
plt.xlabel("Revenue")
plt.ylabel("Frequency")

plt.show()

In [ ]:
plt.figure(figsize=(10, 4))

plt.boxplot(sales_df["Revenue"], vert=False)

plt.title("Revenue Boxplot")
plt.xlabel("Revenue")

plt.show()

In [ ]:
Q1 = sales_df["Revenue"].quantile(0.25)
Q3 = sales_df["Revenue"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = sales_df[
    (sales_df["Revenue"] < lower_bound) |
    (sales_df["Revenue"] > upper_bound)
]

print("Q1:", round(Q1, 2))
print("Q3:", round(Q3, 2))
print("IQR:", round(IQR, 2))
print("Lower Bound:", round(lower_bound, 2))
print("Upper Bound:", round(upper_bound, 2))
print("Number of Revenue Outliers:", len(outliers))
print("Outlier Percentage:", round(len(outliers) / len(sales_df) * 100, 2), "%")

In [ ]:
revenue_outliers = sales_df[sales_df["Revenue"] > upper_bound]

print("Revenue outlier transactions:", len(revenue_outliers))

print("\nTop 10 highest-revenue transactions:")
display(
    revenue_outliers[
        ["InvoiceNo", "Description", "Quantity", "UnitPrice", "Revenue", "Country"]
    ]
    .sort_values("Revenue", ascending=False)
    .head(10)
)

In [ ]:
print("Revenue Outlier Analysis")
print("------------------------")

print("Outliers with Quantity > 100:",
      (revenue_outliers["Quantity"] > 100).sum())

print("Outliers with UnitPrice > 100:",
      (revenue_outliers["UnitPrice"] > 100).sum())

print("Outliers with Quantity <= 100:",
      (revenue_outliers["Quantity"] <= 100).sum())

print("Outliers with UnitPrice <= 100:",
      (revenue_outliers["UnitPrice"] <= 100).sum())

In [ ]:
sales_df["Month"] = sales_df["InvoiceDate"].dt.to_period("M").astype(str)

monthly_sales = (
    sales_df.groupby("Month", as_index=False)["Revenue"]
    .sum()
)

print("Monthly revenue data created!")
print("Number of months:", len(monthly_sales))

display(monthly_sales.head())

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(monthly_sales["Month"], monthly_sales["Revenue"], marker="o")

plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Total Revenue")

plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)

plt.show()

In [ ]:
highest_month = monthly_sales.loc[monthly_sales["Revenue"].idxmax()]
lowest_month = monthly_sales.loc[monthly_sales["Revenue"].idxmin()]

print("Highest Revenue Month:")
print("Month:", highest_month["Month"])
print("Revenue:", round(highest_month["Revenue"], 2))

print("\nLowest Revenue Month:")
print("Month:", lowest_month["Month"])
print("Revenue:", round(lowest_month["Revenue"], 2))

In [ ]:
top_products = (
    sales_df.groupby("Description")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print("Top 10 Products by Revenue:")
display(top_products)

In [ ]:
product_sales = (
    sales_df.groupby("Description", as_index=False)
    .agg(
        Total_Revenue=("Revenue", "sum"),
        Units_Sold=("Quantity", "sum"),
        Transactions=("InvoiceNo", "nunique")
    )
    .sort_values("Total_Revenue", ascending=False)
)

# Remove obvious non-product/service entries
non_product_terms = [
    "POSTAGE",
    "DOTCOM POSTAGE",
    "Manual",
    "AMAZON FEE",
    "BANK CHARGES",
    "ADJUST"
]

product_sales_filtered = product_sales[
    ~product_sales["Description"].str.upper().str.contains(
        "|".join(non_product_terms), na=False
    )
]

print("Top 10 actual products by revenue:")
display(product_sales_filtered.head(10))

In [ ]:
non_product_terms = [
    "POSTAGE",
    "DOTCOM POSTAGE",
    "MANUAL",
    "AMAZON FEE",
    "BANK CHARGES",
    "ADJUST"
]

product_sales_filtered = product_sales[
    ~product_sales["Description"].str.upper().str.contains(
        "|".join(non_product_terms), na=False
    )
]

print("Top 10 actual products by revenue:")
display(product_sales_filtered.head(10))

In [ ]:
top_10_products = product_sales_filtered.head(10).sort_values(
    "Total_Revenue", ascending=True
)

plt.figure(figsize=(10, 6))

plt.barh(
    top_10_products["Description"],
    top_10_products["Total_Revenue"]
)

plt.title("Top 10 Products by Revenue")
plt.xlabel("Total Revenue")
plt.ylabel("Product")

plt.tight_layout()
plt.show()

In [ ]:
country_sales = (
    sales_df.groupby("Country", as_index=False)
    .agg(
        Total_Revenue=("Revenue", "sum"),
        Orders=("InvoiceNo", "nunique")
    )
    .sort_values("Total_Revenue", ascending=False)
)

print("Top 10 Countries by Revenue:")
display(country_sales.head(10))

In [ ]:
top_10_countries = country_sales.head(10).sort_values(
    "Total_Revenue", ascending=True
)

plt.figure(figsize=(10, 6))

plt.barh(
    top_10_countries["Country"],
    top_10_countries["Total_Revenue"]
)

plt.title("Top 10 Countries by Revenue")
plt.xlabel("Total Revenue")
plt.ylabel("Country")

plt.tight_layout()
plt.show()

In [ ]:
correlation_matrix = sales_df[
    ["Quantity", "UnitPrice", "Revenue"]
].corr()

print("Correlation Matrix:")
display(correlation_matrix.round(3))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 5))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".3f",
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(
    sales_df["Quantity"],
    sales_df["Revenue"],
    alpha=0.3
)

plt.title("Quantity vs Revenue")
plt.xlabel("Quantity")
plt.ylabel("Revenue")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(
    sales_df["Quantity"],
    sales_df["Revenue"],
    alpha=0.25
)

plt.xscale("log")
plt.yscale("log")

plt.title("Quantity vs Revenue — Log Scale")
plt.xlabel("Quantity (log scale)")
plt.ylabel("Revenue (log scale)")

plt.tight_layout()
plt.show()

In [ ]:
total_revenue = sales_df["Revenue"].sum()
total_units = sales_df["Quantity"].sum()
total_orders = sales_df["InvoiceNo"].nunique()
unique_products = sales_df["Description"].nunique()
unique_countries = sales_df["Country"].nunique()

print("KEY BUSINESS KPIs")
print("------------------")
print("Total Revenue:", round(total_revenue, 2))
print("Total Units Sold:", total_units)
print("Total Orders:", total_orders)
print("Unique Products:", unique_products)
print("Countries Served:", unique_countries)

In [ ]:
average_order_value = total_revenue / total_orders

print("Average Order Value:", round(average_order_value, 2))

In [ ]:
average_units_per_order = total_units / total_orders

print("Average Units per Order:", round(average_units_per_order, 2))

In [ ]:
uk_revenue = country_sales.loc[
    country_sales["Country"] == "United Kingdom",
    "Total_Revenue"
].iloc[0]

uk_share = (uk_revenue / total_revenue) * 100

print("UK Revenue Share:", round(uk_share, 2), "%")

# Key Findings & Business Implications

## Key Findings

* The dataset contains **10.64 million in total recorded revenue** across **19,960 orders**, **4,026 products**, and **38 countries**.
* The **United Kingdom accounts for 84.59% of total revenue**, showing a strong concentration in the UK market.
* Monthly revenue was lowest in **February 2011 (522,545.56)** and highest in **November 2011 (1,503,866.78)**.
* Revenue is **highly right-skewed**, with a skewness value of **504.23**. The mean revenue (**20.28**) is substantially higher than the median (**9.92**).
* The IQR method identified **42,624 revenue outliers (8.12%)**, with the upper outlier threshold at **38.40**.
* Several extreme transactions involve unusually large quantities. For example, one transaction recorded **80,995 units**, demonstrating the importance of investigating extreme observations before treating them as errors.
* Among the filtered actual products, **REGENCY CAKESTAND 3 TIER** generated the highest recorded revenue at **174,156.54**.
* Quantity and revenue show a strong positive correlation (**r = 0.907**). However, this relationship is partly mechanical because revenue was calculated as **Quantity × UnitPrice**.
* Unit price and revenue show a weak positive correlation (**r = 0.137**).
* Data-quality issues were investigated systematically, including duplicates, missing descriptions, missing CustomerIDs, returns/cancellations, and non-standard accounting or service entries.

## Business Implications

* The high UK revenue share suggests that geographic concentration should be monitored when evaluating market exposure and diversification opportunities.
* The increase in revenue during September–November provides an opportunity to investigate seasonal purchasing patterns, promotions, and holiday-related demand.
* Extreme transactions should be reviewed separately because they may represent legitimate bulk purchases rather than data errors.
* Product-level revenue can support further investigation of consistently strong products and products whose results are driven by unusual transactions.
* Separating sales, returns, postage, fees, and accounting adjustments can improve the accuracy of future business reporting and dashboards.


# Executive Summary

## Problem

This project examines e-commerce transaction data to identify sales patterns, product performance, geographic concentration, trends, and unusual transactions. The objective is to transform raw transaction records into meaningful business insights through Exploratory Data Analysis (EDA).

## Methodology

The dataset was cleaned and investigated using Python and Pandas. Duplicate records and incomplete product descriptions were addressed, while missing CustomerIDs were retained for transaction-level analysis. Returns, cancellations, and accounting-related entries were investigated separately from valid positive sales. Revenue was calculated as Quantity × UnitPrice. Descriptive statistics, skewness analysis, histograms, boxplots, IQR-based outlier detection, time-series analysis, product and country analysis, correlation analysis, and scatter plots were used to identify patterns and anomalies.

## Key Findings

The analysis recorded total revenue of **10.64 million** across **19,960 orders**, **4,026 products**, and **38 countries**. The UK generated **84.59% of total revenue**, indicating strong geographic concentration. Revenue increased substantially during September–November 2011, reaching its highest monthly value of **1,503,866.78 in November 2011**.

Revenue was highly right-skewed, with a skewness of **504.23**, and the IQR method identified **42,624 transactions (8.12%)** as revenue outliers. Some extreme observations were associated with exceptionally large quantities, highlighting the importance of investigating unusual transactions before removing them. Quantity and revenue showed a strong positive correlation (**r = 0.907**), while UnitPrice and revenue showed a weak positive correlation (**r = 0.137**).

## Recommendations

The business should monitor its high dependence on the UK market, investigate seasonal revenue patterns, and review extreme transactions separately from normal sales activity. Product-level performance should also be evaluated using both revenue and transaction frequency to distinguish consistently strong products from unusual bulk transactions. Future reporting can be improved by maintaining separate classifications for sales, returns, postage, fees, and accounting adjustments.
